# Módulo 5 · Notebook 1 — Tools para Agentes

Antes de construir agentes completos, necesitamos entender su bloque básico: las **tools**.

Una tool es una función que el modelo puede decidir usar cuando necesita:
- consultar datos externos,
- hacer cálculos,
- ejecutar lógica determinística.

En este notebook vas a:
1. Crear tools con `@tool`.
2. Probarlas directamente.
3. Ver cómo un LLM decide llamar tools (tool-calling).

## 1. Setup

In [1]:
import warnings
from statistics import mean

from langchain_core.tools import tool
from langchain_ollama import ChatOllama

warnings.filterwarnings("ignore")

# Dataset pequeño para demos de herramientas
WORLD_CUP_CHAMPIONS = {
    1930: "Uruguay",
    1934: "Italia",
    1938: "Italia",
    1950: "Uruguay",
    1954: "Alemania Federal",
    1958: "Brasil",
    1962: "Brasil",
    1966: "Inglaterra",
    1970: "Brasil",
    1974: "Alemania Federal",
    1978: "Argentina",
    1982: "Italia",
    1986: "Argentina",
    1990: "Alemania",
    1994: "Brasil",
    1998: "Francia",
    2002: "Brasil",
    2006: "Italia",
    2010: "España",
}

GOALS_PER_WORLD_CUP = {
    1930: 70,
    1934: 70,
    1938: 84,
    1950: 88,
    1954: 140,
    1958: 126,
    1962: 89,
    1966: 89,
    1970: 95,
    1974: 97,
    1978: 102,
    1982: 146,
    1986: 132,
    1990: 115,
    1994: 141,
    1998: 171,
    2002: 161,
    2006: 147,
    2010: 145,
}

llm = ChatOllama(model="llama3.2:3b", temperature=0)
print("✅ Entorno listo")

c:\_dev\IA\AI-course\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Entorno listo


## 2. Definir tools

Buenas prácticas:
- Nombre claro de la función.
- Docstring corto y preciso (el modelo la usa para decidir cuándo llamar la tool).
- Entrada/salida simple y tipada.

In [2]:
@tool
def campeon_del_mundial(anio: int) -> str:
    """Devuelve el campeón del Mundial para un año específico."""
    return WORLD_CUP_CHAMPIONS.get(anio, f"No tengo dato111s del año {anio}.")


@tool
def promedio_goles(inicio: int, fin: int) -> str:
    """Calcula el promedio de goles entre dos mundiales (años incluidos)."""
    years = [y for y in GOALS_PER_WORLD_CUP if inicio <= y <= fin]
    if not years:
        return "No hay mundiales en ese rango."
    avg = mean(GOALS_PER_WORLD_CUP[y] for y in years)
    return f"Promedio de goles entre {inicio} y {fin}: {avg:.2f}"


@tool
def listar_campeones() -> str:
    """Lista los campeones de los mundiales disponibles en el dataset."""
    lines = [f"{y}: {WORLD_CUP_CHAMPIONS[y]}" for y in sorted(WORLD_CUP_CHAMPIONS)]
    return "\n".join(lines)


tools = [campeon_del_mundial, promedio_goles, listar_campeones]
print("Tools creadas:", [t.name for t in tools])

Tools creadas: ['campeon_del_mundial', 'promedio_goles', 'listar_campeones']


## 3. Probar tools directamente

In [3]:
print(campeon_del_mundial.invoke({"anio": 1998}))
print(promedio_goles.invoke({"inicio": 1990, "fin": 2010}))
print(listar_campeones.invoke({})[:180], "...")

Francia
Promedio de goles entre 1990 y 2010: 146.67
1930: Uruguay
1934: Italia
1938: Italia
1950: Uruguay
1954: Alemania Federal
1958: Brasil
1962: Brasil
1966: Inglaterra
1970: Brasil
1974: Alemania Federal
1978: Argentina
1982: It ...


## 4. Tool-calling con el LLM

Aquí no hay agente completo todavía. Solo dejamos que el LLM decida si necesita una tool.

In [4]:
llm_with_tools = llm.bind_tools(tools)

question = "¿Quién ganó el mundial de 1986?"
ai_msg = llm_with_tools.invoke(question)

print("Contenido del modelo:", ai_msg.content)
print("Tool calls:", ai_msg.tool_calls)

if ai_msg.tool_calls:
    # Ejecutamos manualmente la primera llamada para ver el ciclo básico
    call = ai_msg.tool_calls[0]
    selected_tool = {
        "campeon_del_mundial": campeon_del_mundial,
        "promedio_goles": promedio_goles,
        "listar_campeones": listar_campeones,
    }[call["name"]]
    tool_result = selected_tool.invoke(call["args"])
    print("Resultado de tool:", tool_result)
else:
    print("El modelo respondió sin usar tools.")

Contenido del modelo: 
Tool calls: [{'name': 'campeon_del_mundial', 'args': {'anio': 1986}, 'id': '62019645-1e3a-4fa2-a122-ffbdf568003a', 'type': 'tool_call'}]
Resultado de tool: Argentina


## 5. Resumen

- Las tools son funciones con interfaz clara para que el modelo las use.
- `@tool` convierte una función Python en una herramienta compatible con LangChain.
- `bind_tools` habilita la decisión de llamar herramientas.
- En el siguiente notebook automatizamos el ciclo completo con un agente ReAct.